# Notebook 03: Transformación y Validación de Calidad
## Proyecto II Parcial — Modelado Avanzado de Base de Datos - 30759
## Integrantes:
- Naomi Obando
- Mauri Tandazo

**Fases 4-5:** Transformación de datos (Silver) + validación de calidad y separación de rechazados.

---
### Campos derivados que se calculan
| Campo | Fórmula |
|---|---|
| `trip_duration_minutes` | `(dropoff - pickup) / 60` |
| `average_speed_mph` | `trip_distance / (duration_h)` |
| `fare_per_mile` | `fare_amount / trip_distance` |
| `tip_percentage` | `(tip_amount / fare_amount) × 100` |
| `is_airport_trip` | pickup_location_id o dropoff_location_id in [1, 132, 138] |
| `is_suspicious_trip` | OR de múltiples condiciones anómalas |
| `trip_id` | SHA-256 de campos clave (idempotente) |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pyspark.sql import functions as F
from src.utils          import generate_process_id, load_config, setup_logger, create_spark_session
from src.schema_recovery import read_bronze_layer
from src.transformations  import (
    run_transformation_phase, save_silver_layer,
    add_trip_id, add_trip_duration, add_average_speed,
    add_fare_per_mile, add_tip_percentage, add_suspicious_flag
)
from src.quality_rules   import run_quality_phase, apply_quality_rules

PROCESS_ID  = generate_process_id()
CONFIG_PATH = '../config/etl_config.yaml'
config      = load_config(CONFIG_PATH)
config['paths'] = {k: f'../{v}' for k, v in config['paths'].items()}
config['paths']['metadata_dir'] = '../metadata'

logger = setup_logger('nb03_transform', '../logs', PROCESS_ID)
spark  = create_spark_session(config)
print(f'process_id: {PROCESS_ID}')

process_id: ETL-20260623-110857-C78E7A45


In [2]:
# ── 1. Leer capa Bronze ───────────────────────────────────────────
bronze_df = read_bronze_layer(spark, config['paths']['bronze_dir'], logger)

if bronze_df is None:
    print('⚠️ Bronze no encontrado. Ejecuta primero el Notebook 02.')
else:
    bronze_count = bronze_df.count()
    print(f'Bronze cargado: {bronze_count:,} registros, {len(bronze_df.columns)} columnas')
    bronze_df.groupBy('service_type').count().show()

2026-06-23 11:09:14 | INFO     | nb03_transform                 | Capa bronze cargada: 44,502,927 registros, 27 columnas
Bronze cargado: 44,502,927 registros, 27 columnas
+------------+--------+
|service_type|   count|
+------------+--------+
|       fhvhv|18479031|
|      yellow|25890876|
|       green|  133020|
+------------+--------+



In [3]:
# ── 2. Demostración de transformaciones individuales ──────────────
print('=== DEMOSTRACIÓN DE CAMPOS DERIVADOS ===')

# Calcular paso a paso para mostrar cada transformación
demo_df = (
    bronze_df.limit(5)
    .transform(add_trip_id)
    .transform(add_trip_duration)
    .transform(add_average_speed)
    .transform(add_fare_per_mile)
    .transform(add_tip_percentage)
)

demo_df.select(
    'service_type', 'pickup_datetime', 'dropoff_datetime',
    'trip_distance',
    'trip_duration_minutes',
    'average_speed_mph',
    'fare_per_mile',
    'tip_percentage',
    F.col('trip_id').substr(1, 16).alias('trip_id_short')
).show(truncate=False)

=== DEMOSTRACIÓN DE CAMPOS DERIVADOS ===
+------------+-------------------+-------------------+-------------+---------------------+-----------------+-------------+--------------+----------------+
|service_type|pickup_datetime    |dropoff_datetime   |trip_distance|trip_duration_minutes|average_speed_mph|fare_per_mile|tip_percentage|trip_id_short   |
+------------+-------------------+-------------------+-------------+---------------------+-----------------+-------------+--------------+----------------+
|fhvhv       |2023-01-04 23:26:21|2023-01-04 23:28:54|0.7          |2.55                 |16.47            |11.3         |0.0           |0256fb975fe94fd4|
|fhvhv       |2023-01-31 23:33:52|2023-01-31 23:53:03|8.63         |19.18                |27.0             |2.6837       |0.0           |35121b75d7922566|
|fhvhv       |2023-01-04 23:03:45|2023-01-04 23:18:54|7.76         |15.15                |30.73            |2.0116       |0.0           |ca0a460a7fcc282a|
|fhvhv       |2023-01-31 23:0

In [4]:
# ── 3. Mostrar el plan Spark (explain) — Optimización 5 ──────────
print('=== PLAN DE EJECUCIÓN SPARK (explain) ===')
print('Optimización: Spark empuja predicates al lector Parquet (predicate pushdown)\n')

# Filtro de muestra para ver pushdown en acción
sample = bronze_df.filter(
    (F.col('service_type') == 'yellow') & (F.col('year') == 2023)
).select('pickup_datetime', 'total_amount', 'trip_distance')

sample.explain(mode='formatted')

=== PLAN DE EJECUCIÓN SPARK (explain) ===
Optimización: Spark empuja predicates al lector Parquet (predicate pushdown)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [pickup_datetime#1, trip_distance#4, total_amount#16, service_type#24, year#25, month#26]
Batched: true
Location: InMemoryFileIndex [file:/D:/etl_spark_parquet_advanced/data/bronze]
PartitionFilters: [isnotnull(service_type#24), isnotnull(year#25), (service_type#24 = yellow), (year#25 = 2023)]
ReadSchema: struct<pickup_datetime:timestamp,trip_distance:double,total_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [pickup_datetime#1, trip_distance#4, total_amount#16, service_type#24, year#25, month#26]

(3) Project [codegen id : 1]
Output [3]: [pickup_datetime#1, total_amount#16, trip_distance#4]
Input [6]: [pickup_datetime#1, trip_distance#4, total_amount#16, service_type#24, year#25, month#26]




In [5]:
# ── 4. Ejecutar transformación completa ───────────────────────────
silver_df = run_transformation_phase(spark, bronze_df, config, PROCESS_ID, logger)
save_silver_layer(silver_df, config['paths']['silver_dir'], config, logger)

print(f'\n✅ Silver construido: {silver_df.count():,} registros')
silver_df.printSchema()

2026-06-23 11:09:19 | INFO     | nb03_transform                 | ══════════════════════════════════════════════════════════════════════
2026-06-23 11:09:19 | INFO     | nb03_transform                 |   FASE 4: TRANSFORMACIÓN  —  process_id: ETL-20260623-110857-C78E7A45
2026-06-23 11:09:19 | INFO     | nb03_transform                 | ══════════════════════════════════════════════════════════════════════
2026-06-23 11:09:19 | INFO     | nb03_transform                 | [4.0] Registros entrantes: 44,502,927
2026-06-23 11:09:19 | INFO     | nb03_transform                 | [4.1]  Normalizando strings y montos...
2026-06-23 11:09:19 | INFO     | nb03_transform                 | [4.2]  Recalculando particiones year/month desde fechas reales...
2026-06-23 11:09:19 | INFO     | nb03_transform                 | [4.3]  Generando trip_id (SHA-256)...
2026-06-23 11:09:20 | INFO     | nb03_transform                 | [4.4]  Calculando duración del viaje (minutos)...
2026-06-23 11:09:20 | INFO  

In [6]:
# ── 5. Análisis de viajes sospechosos ────────────────────────────
print('=== ANÁLISIS DE VIAJES SOSPECHOSOS ===')

suspicious_stats = (
    silver_df
    .groupBy('service_type', 'is_suspicious_trip')
    .agg(
        F.count('*').alias('count'),
        F.round(F.avg('trip_distance'), 2).alias('avg_distance'),
        F.round(F.avg('total_amount'), 2).alias('avg_total')
    )
    .orderBy('service_type', 'is_suspicious_trip')
)
suspicious_stats.show()

print('\n=== EJEMPLOS DE VIAJES SOSPECHOSOS ===')
silver_df.filter(F.col('is_suspicious_trip') == True) \
    .select('service_type','trip_duration_minutes','average_speed_mph',
            'trip_distance','total_amount','tip_percentage') \
    .limit(5).show(truncate=False)

=== ANÁLISIS DE VIAJES SOSPECHOSOS ===
+------------+------------------+--------+------------+---------+
|service_type|is_suspicious_trip|   count|avg_distance|avg_total|
+------------+------------------+--------+------------+---------+
|       fhvhv|              NULL|      12|        1.41|     2.25|
|       fhvhv|             false|18452963|        4.87|     27.4|
|       fhvhv|              true|   26055|        3.73|    14.49|
|       green|              NULL|       4|        7.88|    13.39|
|       green|             false|  125636|        2.86|    21.79|
|       green|              true|    7372|      146.78|    22.71|
|      yellow|              NULL|    1540|        6.22|     8.46|
|      yellow|             false|25312550|        3.23|    23.22|
|      yellow|              true|  562740|       42.31|    16.54|
+------------+------------------+--------+------------+---------+


=== EJEMPLOS DE VIAJES SOSPECHOSOS ===
+------------+---------------------+-----------------+--------

In [7]:
# ── 6. Aplicar reglas de calidad y ver rechazados ─────────────────
print('=== VALIDACIÓN DE CALIDAD ===')
valid_df, rejected_df = apply_quality_rules(silver_df, config, PROCESS_ID, logger)

total    = silver_df.count()
valid    = valid_df.count()
rejected = rejected_df.count()

print(f'Total procesados : {total:,}')
print(f'Válidos          : {valid:,} ({valid/total*100:.2f}%)')
print(f'Rechazados       : {rejected:,} ({rejected/total*100:.2f}%)')

print('\n=== MOTIVOS DE RECHAZO ===')
rejected_df.groupBy('rejection_category', 'rejection_rule') \
    .count().orderBy(F.col('count').desc()).show(truncate=False)

=== VALIDACIÓN DE CALIDAD ===
2026-06-23 11:30:28 | INFO     | nb03_transform                 |   Aplicando 11 reglas de calidad...
2026-06-23 11:32:45 | INFO     | nb03_transform                 |   Válidos    : 44,250,454
2026-06-23 11:49:23 | INFO     | nb03_transform                 |   Rechazados : 430,582
Total procesados : 44,488,872
Válidos          : 44,250,454 (99.46%)
Rechazados       : 430,582 (0.97%)

=== MOTIVOS DE RECHAZO ===
+------------------+--------------------+------+
|rejection_category|rejection_rule      |count |
+------------------+--------------------+------+
|NEGATIVE_AMOUNT   |negative_fare_amount|177664|
|INVALID_AMOUNT    |invalid_total_amount|176687|
|INVALID_DURATION  |invalid_duration    |54860 |
|INVALID_DATE_ORDER|invalid_date_order  |21286 |
|OUT_OF_RANGE_DATE |out_of_range_year   |85    |
+------------------+--------------------+------+



In [8]:
# ── 7. Estadísticas de calidad por servicio ───────────────────────
print('=== CALIDAD POR SERVICIO Y MES ===')
valid_df.groupBy('service_type', 'year', 'month') \
    .agg(
        F.count('*').alias('registros'),
        F.round(F.sum('total_amount'), 0).alias('ingresos_totales'),
        F.round(F.avg('trip_duration_minutes'), 1).alias('duracion_prom_min'),
        F.round(F.avg('average_speed_mph'), 1).alias('velocidad_prom_mph')
    ) \
    .orderBy('service_type', 'year', 'month') \
    .show(truncate=False)

print(f'\n✅ Fase 4-5 completada — process_id: {PROCESS_ID}')
silver_df.unpersist()

=== CALIDAD POR SERVICIO Y MES ===
+------------+----+-----+---------+----------------+-----------------+------------------+
|service_type|year|month|registros|ingresos_totales|duracion_prom_min|velocidad_prom_mph|
+------------+----+-----+---------+----------------+-----------------+------------------+
|fhvhv       |2023|1    |18465479 |5.06049833E8    |18.2             |14.1              |
|green       |2022|12   |2        |52.0            |24.5             |18.3              |
|green       |2023|1    |67700    |1482161.0       |14.0             |43.9              |
|green       |2023|2    |64252    |1413519.0       |14.2             |62.6              |
|green       |2023|3    |1        |8.0             |2.5              |19.1              |
|yellow      |2019|12   |109      |1932.0          |12.0             |13.5              |
|yellow      |2020|1    |6353508  |1.18975218E8    |13.1             |13.2              |
|yellow      |2020|2    |30       |925.0           |12.6         

DataFrame[vendor_id: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: double, trip_distance: double, pickup_location_id: bigint, dropoff_location_id: bigint, rate_code_id: double, store_and_fwd_flag: string, payment_type: bigint, fare_amount: double, extra_amount: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double, ehail_fee: double, trip_type: double, source_file: string, ingestion_timestamp: timestamp, quality_status: string, service_type: string, year: int, month: int, trip_id: string, trip_duration_minutes: double, average_speed_mph: double, fare_per_mile: double, tip_percentage: double, is_airport_trip: boolean, is_suspicious_trip: boolean, processing_date: date]